In [ ]:
import statistics

import pandas as pd

from etl import *
from regression import *
from sims_engine import *

In [2]:
pd.set_option('display.max_columns', None)

In [3]:
%load_ext autoreload
%autoreload 2

In [4]:
CONFIG = {
    'elo_date': '2025-11-30',  # date for elo ratings from clubelo.com
    'opta_date': '2025-11-28',  # date for elo ratings from the opta -> clubelo regression; elo_date from day X is before the games are played, for opta it depends
    'number_of_sims': 10000,
    'league_id': 106,
    'season': 2025,
    'head_size': 36,
    'country_code_elo': None,  # use this attr. to use elo ratings from clubelo.com
    'country_code_api': 'POL',  # use this attr. to use elo ratings from the opta -> clubelo regression
    'stdev': 0,
    'update_fixtures': True,
    'is_european_league': False,
    'round_no': 18,
}
code = CONFIG['country_code_elo'] if CONFIG['country_code_elo'] is not None else CONFIG['country_code_api']
CONFIG['sorting_order'] = get_sorting_order_for_country_code(code)

In [5]:
# download_elo_data(CONFIG['elo_date'])

In [6]:
# main_regression(**CONFIG)

In [7]:
standings_df = build_historical_standings_table_after_at_most_n_rounds(**CONFIG)
standings_df.head(CONFIG['head_size'])

,Club,Elo,Matches played,Wins,Draws,Losses,Goals for,Goals against,Goal difference,Points,Random order
1,Wisla Plock,1443.38,18,7,9,2,21,12,9,30,11
2,Gornik Zabrze,1513.38,18,9,3,6,29,24,5,30,8
3,Raków Częstochowa,1549.50,17,9,2,6,26,22,4,29,2
4,Jagiellonia,1529.18,16,8,4,4,28,20,8,28,1
5,Cracovia Krakow,1481.76,18,7,6,5,25,21,4,27,3
6,Radomiak Radom,1420.80,18,7,5,6,35,30,5,26,9
7,Lech Poznan,1520.15,17,6,8,3,29,26,3,26,14
8,Zaglebie Lubin,1414.02,17,6,7,4,30,24,6,25,0
9,Korona Kielce,1452.41,18,6,6,6,21,19,2,24,5
10,Pogon Szczecin,1465.96,18,6,3,9,28,32,-4,21,12


In [9]:
with open('data/optimized_elos_2025_2026.json', 'r') as f:
    optimized_elos = json.load(f)

team_map = pd.read_excel('teams_mapping/team_names.xlsx')
team_map = team_map[['fixtures_name', 'football-data_name']]
team_map.dropna(inplace=True)
team_map_dict = dict(zip(team_map['football-data_name'], team_map['fixtures_name']))
optimized_elos_mapped = {
    team_map_dict.get(team, team): elo for team, elo in optimized_elos.items()
}
standings_df['Elo'] = standings_df['Club'].map(optimized_elos_mapped)
standings_df.head(CONFIG['head_size'])

,Club,Elo,Matches played,Wins,Draws,Losses,Goals for,Goals against,Goal difference,Points,Random order
1,Wisla Plock,1483.49,18,7,9,2,21,12,9,30,11
2,Gornik Zabrze,1551.61,18,9,3,6,29,24,5,30,8
3,Raków Częstochowa,1605.27,17,9,2,6,26,22,4,29,2
4,Jagiellonia,1552.33,16,8,4,4,28,20,8,28,1
5,Cracovia Krakow,1521.13,18,7,6,5,25,21,4,27,3
6,Radomiak Radom,1470.72,18,7,5,6,35,30,5,26,9
7,Lech Poznan,1585.70,17,6,8,3,29,26,3,26,14
8,Zaglebie Lubin,1445.62,17,6,7,4,30,24,6,25,0
9,Korona Kielce,1522.17,18,6,6,6,21,19,2,24,5
10,Pogon Szczecin,1521.29,18,6,3,9,28,32,-4,21,12


In [10]:
CONFIG['update_fixtures'] = False

In [11]:
float(round(standings_df['Points'].sum() / standings_df['Matches played'].sum(), 2))

1.34

In [12]:
sample_season = simulate_season_after_n_rounds(**CONFIG, standings_df=standings_df)
sample_season.head(CONFIG['head_size'])

,Club,Elo,Matches played,Wins,Draws,Losses,Goals for,Goals against,Goal difference,Points,Random order
1,Cracovia Krakow,1521.13,34,18,8,8,49,29,20,62,13
2,Jagiellonia,1552.33,34,17,8,9,50,34,16,59,8
3,Raków Częstochowa,1605.27,34,18,5,11,47,35,12,59,12
4,Wisla Plock,1483.49,34,13,14,7,38,27,11,53,4
5,Legia Warszawa,1625.11,34,14,10,10,42,31,11,52,6
6,Korona Kielce,1522.17,34,14,9,11,40,32,8,51,10
7,Lech Poznan,1585.70,34,12,14,8,47,42,5,50,17
8,Gornik Zabrze,1551.61,34,14,5,15,41,44,-3,47,2
9,Radomiak Radom,1470.72,34,12,10,12,50,47,3,46,11
10,Zaglebie Lubin,1445.62,34,11,11,12,44,44,0,44,9


In [13]:
float(round(sample_season['Points'].sum() / sample_season['Matches played'].sum(), 2))

1.36

In [14]:
simulate_odds(**CONFIG, standings_df=standings_df).head(CONFIG['head_size'])

,Home Team,Away Team,Odds H,Odds D,Odds A
0,Lechia Gdansk,Gornik Zabrze,2.74,3.43,2.91
1,Arka Gdynia,Motor Lublin,2.74,3.43,2.91
2,Pogon Szczecin,Radomiak Radom,2.06,3.72,4.08
3,Zaglebie Lubin,Widzew Łódź,3.03,3.44,2.64
4,Piast Gliwice,Legia Warszawa,3.27,3.47,2.46
5,Nieciecza,Jagiellonia,3.53,3.54,2.30
6,Cracovia Krakow,Lech Poznan,2.79,3.42,2.86
7,Raków Częstochowa,GKS Katowice,1.69,4.39,5.56
8,Korona Kielce,Wisla Plock,2.12,3.66,3.93


In [ ]:
# full table sim
results = run_full_table_sims(**CONFIG, standings_df=standings_df, fixtures_matrix=fixtures_matrix)
results.head(CONFIG['head_size'])

In [ ]:
# removed: run_multiple_sims for top 1 — use position columns from run_full_table_sims output instead

In [ ]:
# removed: run_multiple_sims for top 2 — use position columns from run_full_table_sims output instead

In [ ]:
# removed: run_multiple_sims for top 3 — use position columns from run_full_table_sims output instead

In [ ]:
# removed: run_multiple_sims for top 4 — use position columns from run_full_table_sims output instead

In [ ]:
# removed: run_multiple_sims for bottom 3 — use position columns from run_full_table_sims output instead

In [ ]:
# removed: run_multiple_sims for bottom 1 — use position columns from run_full_table_sims output instead